In [1]:
import os
import numpy as np
import pandas as pd

# Relative to Our Notebooks/
PROVIDED_DIR = "../Provided Datasets"
NEW_DIR = "../New Datasets"
OUTPUT_DIR = NEW_DIR + "/Combined"
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
# Expected training files
EXPECTED_FILES = {
    "gaia": "gaia_features_training.csv",
    "jrc_gsw": "jrc_gsw_features_training.csv",
    "landsat_allbands": "landsat_features_training_allbands.csv",
    "terraclimate_allvars": "terraclimate_features_training_allvars.csv",
    "esa_cci": "esa_cci_features_training.csv",
    "raster_buffer": "esa_jrc_gaia_buffer_training.csv",
}

def resolve_path(fname: str) -> str | None:
    """Return the first existing path for fname across Provided and New dirs."""
    for base in (PROVIDED_DIR, NEW_DIR):
        p = os.path.join(base, fname)
        if os.path.exists(p):
            return p
    return None

resolved = {}
missing = []
for key, fname in EXPECTED_FILES.items():
    p = resolve_path(fname)
    if p is None:
        missing.append((key, fname))
    else:
        resolved[key] = p

print("Resolved input paths:")
for k, p in resolved.items():
    print(f" - {k:18s} -> {os.path.abspath(p)}")

if missing:
    msg = "Missing these expected files in BOTH folders:\n" + "\n".join([f"{k}: {f}" for k, f in missing])
    raise FileNotFoundError(msg)

print("\nAll expected files found")

Resolved input paths:
 - gaia               -> /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/gaia_features_training.csv
 - jrc_gsw            -> /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/jrc_gsw_features_training.csv
 - landsat_allbands   -> /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/landsat_features_training_allbands.csv
 - terraclimate_allvars -> /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/terraclimate_features_training_allvars.csv
 - esa_cci            -> /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/esa_cci_features_training.csv
 - raster_buffer      -> /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/esa_jrc_gaia_buffer_training.csv

All expected files found


In [3]:
def standardize_join_keys(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize join columns to: latitude, longitude, sample_date."""
    out = df.copy()
    rename_map = {}
    for c in out.columns:
        lc = c.strip().lower()
        if lc in {"latitude", "lat"} or lc.startswith("lat"):
            rename_map[c] = "latitude"
        elif lc in {"longitude", "lon", "lng"} or lc.startswith("lon"):
            rename_map[c] = "longitude"
        # Be conservative: only map obvious sample-date columns
        elif lc in {"sample date", "sample_date"}:
            rename_map[c] = "sample_date"
    out = out.rename(columns=rename_map)

    # If a dataset used a generic "Date" column, map it ONLY if sample_date doesn't exist yet
    if "sample_date" not in out.columns:
        for c in list(out.columns):
            if c.strip().lower() == "date":
                out = out.rename(columns={c: "sample_date"})
                break

    # Drop duplicate columns created by renaming collisions
    out = out.loc[:, ~out.columns.duplicated(keep="first")]

    required = {"latitude", "longitude", "sample_date"}
    missing = required - set(out.columns)
    if missing:
        raise ValueError(f"Missing required join columns after standardization: {missing}")

    # helper parsed date (not used for join)
    out["sample_date_parsed"] = pd.to_datetime(out["sample_date"], errors="coerce", infer_datetime_format=True)
    return out

In [4]:
# Load + standardize
datasets = {}
for name, path in resolved.items():
    df = pd.read_csv(path)
    df_std = standardize_join_keys(df)
    datasets[name] = df_std
    print(f"{name:18s} shape={df_std.shape}  cols={len(df_std.columns)}")

gaia               shape=(9319, 9)  cols=9
jrc_gsw            shape=(9319, 13)  cols=13
landsat_allbands   shape=(9319, 16)  cols=16
terraclimate_allvars shape=(9319, 18)  cols=18
esa_cci            shape=(9319, 9)  cols=9
raster_buffer      shape=(9319, 26)  cols=26


/var/folders/n3/rr8tb4gs4fd1fst6mp_0w6540000gn/T/ipykernel_48973/2031521441.py:32: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  out["sample_date_parsed"] = pd.to_datetime(out["sample_date"], errors="coerce", infer_datetime_format=True)
/var/folders/n3/rr8tb4gs4fd1fst6mp_0w6540000gn/T/ipykernel_48973/2031521441.py:32: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  out["sample_date_parsed"] = pd.to_datetime(out["sample_date"], errors="coerce", infer_datetime_format=True)
/var/folders/n3/rr8tb4gs4fd1fst6mp_0w6540000gn/T/ipykernel_48973/2031521441.py:32: UserW

In [5]:
# Outer merge on join keys
merge_keys = ["latitude", "longitude", "sample_date"]

merged = None
for name, df in datasets.items():
    if merged is None:
        merged = df
    else:
        merged = pd.merge(
            merged,
            df,
            on=merge_keys,
            how="outer",
            suffixes=("", f"__{name}")  # helps prevent _x/_y
        )

def remove_extra_date_columns(df):
    cols_to_drop = [
        c for c in df.columns
        if ("date" in c.lower()) and (c.lower() != "sample_date")
    ]
    
    print("Dropping from dataframe:")
    print(cols_to_drop)
    
    return df.drop(columns=cols_to_drop)

# Apply to all versions you are saving
merged = remove_extra_date_columns(merged)

if "filled_mean" in globals():
    filled_mean = remove_extra_date_columns(filled_mean)

if "dropped_na" in globals():
    dropped_na = remove_extra_date_columns(dropped_na)

print("Final shape:", merged.shape)

Dropping from dataframe:
['sample_date_parsed', 'sample_date_parsed__jrc_gsw', 'sample_date_parsed__landsat_allbands', 'sample_date_parsed__terraclimate_allvars', 'sample_date_parsed__esa_cci', 'sample_date_parsed__raster_buffer']
Final shape: (9319, 70)


In [6]:
# shape checking
esa = pd.read_csv(os.path.join(PROJECT_ROOT + "/New Datasets", "esa_cci_features_training.csv"))
jrc = pd.read_csv(os.path.join(PROJECT_ROOT + "/New Datasets", "jrc_gsw_features_training.csv"))
gaia = pd.read_csv(os.path.join(PROJECT_ROOT + "/New Datasets", "gaia_features_training.csv"))
landsat = pd.read_csv(os.path.join(PROJECT_ROOT + "/New Datasets", "landsat_features_training_allbands.csv"))
terraclimate = pd.read_csv(os.path.join(PROJECT_ROOT + "/New Datasets", "terraclimate_features_training_allvars.csv"))
real_shape = esa.shape[1] + jrc.shape[1] + gaia.shape[1] + landsat.shape[1] + terraclimate.shape[1]
# subtract 15 for the join keys, then add 3 since we need lat long and date
real_shape = real_shape - 15 + 3
print(f"Real Shape: {real_shape}")
print(f"Merged Shape: {merged.shape[1]}")

# combined validation
# Validate that merged cell values match each source dataset
merge_keys = ["latitude", "longitude", "sample_date"]

def _col_in_merged(col: str, dataset_name: str) -> str | None:
    if col in merge_keys:
        return col
    if col in merged.columns:
        return col
    alt = f"{col}__{dataset_name}"
    if alt in merged.columns:
        return alt
    return None


def _equal_series(a: pd.Series, b: pd.Series, tol: float = 1e-6) -> pd.Series:
    # Treat NaN == NaN as equal; compare numerics with tolerance
    a_num = pd.to_numeric(a, errors="coerce")
    b_num = pd.to_numeric(b, errors="coerce")

    both_nan = a.isna() & b.isna()
    both_num = a_num.notna() & b_num.notna()
    close_num = pd.Series(False, index=a.index)
    close_num[both_num] = np.isclose(a_num[both_num], b_num[both_num], atol=tol, rtol=0)

    # Non-numeric or mixed: compare as strings (but keep NaNs handled already)
    a_str = a.astype(str)
    b_str = b.astype(str)
    str_equal = (a_str == b_str) & (~both_num)

    return both_nan | close_num | str_equal


def validate_dataset(name: str, df_std: pd.DataFrame, tol: float = 1e-6) -> pd.DataFrame:
    # Drop helper parsed date if present
    df_std = df_std.drop(columns=[c for c in df_std.columns if c == "sample_date_parsed"], errors="ignore")

    # Align with merged by keys
    df_std = df_std.copy()
    df_std["_row_id"] = df_std.index
    merged_with = merged.merge(df_std, on=merge_keys, how="left", suffixes=("", "__src"))

    mismatch_rows = []
    for col in df_std.columns:
        if col in merge_keys or col == "_row_id":
            continue
        merged_col = _col_in_merged(col, name)
        if merged_col is None:
            mismatch_rows.append({"dataset": name, "column": col, "issue": "missing_in_merged"})
            continue

        src_col = f"{col}__src"
        if src_col not in merged_with.columns:
            mismatch_rows.append({"dataset": name, "column": col, "issue": "missing_in_source_after_merge"})
            continue

        equal_mask = _equal_series(merged_with[merged_col], merged_with[src_col], tol=tol)
        if not bool(equal_mask.all()):
            mismatch_rows.append({
                "dataset": name,
                "column": col,
                "issue": "value_mismatch",
                "mismatch_count": int((~equal_mask).sum()),
            })

    return pd.DataFrame(mismatch_rows)


mismatch_reports = []
for name, df_std in datasets.items():
    report = validate_dataset(name, df_std)
    mismatch_reports.append(report)

mismatch_summary = pd.concat(mismatch_reports, ignore_index=True) if mismatch_reports else pd.DataFrame()
print("Mismatch summary (empty means all checks passed):")
display(mismatch_summary)

Real Shape: 48
Merged Shape: 70
Mismatch summary (empty means all checks passed):


""


In [7]:
# dropping features
dropped_cols = [
    "qa_pixel",
    "qa_aerosol",
    "esa_processed_flag",
    "esa_observation_count",
    "esa_current_pixel_state",
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus",
    "coastal",
    "lwir11"
]

cols_to_drop = [c for c in dropped_cols if c in merged.columns]
filtered_no_pixel_flags = merged.drop(columns=cols_to_drop).copy()

# Remove raster buffer features from the "normal" dataset
raster_cols = [c for c in filtered_no_pixel_flags.columns if c.endswith("_1km")]
filtered_no_pixel_flags = filtered_no_pixel_flags.drop(columns=raster_cols)

print("Dropped columns:", cols_to_drop)
print("Dropped raster columns:", raster_cols)
print(filtered_no_pixel_flags.shape)

# Save normal (non-engineered) dataset
out_no_flags = os.path.join(OUTPUT_DIR, "combined_training_dataset.csv")
filtered_no_pixel_flags.to_csv(out_no_flags, index=False)
print("Saved:", os.path.abspath(out_no_flags))

Dropped columns: ['qa_pixel', 'qa_aerosol', 'esa_processed_flag', 'esa_observation_count', 'esa_current_pixel_state', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'coastal', 'lwir11']
Dropped raster columns: ['esa_urban_frac_1km', 'esa_cropland_frac_1km', 'esa_water_frac_1km', 'esa_forest_frac_1km', 'esa_shrub_frac_1km', 'esa_grass_frac_1km', 'esa_sparse_veg_frac_1km', 'esa_bare_frac_1km', 'esa_snow_ice_frac_1km', 'esa_flooded_frac_1km', 'esa_other_frac_1km', 'gsw_occurrence_mean_1km', 'gsw_seasonality_mean_1km', 'gsw_recurrence_mean_1km', 'gsw_extent_mean_1km', 'gsw_change_mean_1km', 'gsw_water_frac_1km', 'gaia_changed_ever_frac_1km', 'gaia_impervious_frac_by_sample_year_1km', 'gaia_recent_change_5y_frac_1km', 'gaia_years_since_change_mean_1km', 'gaia_transition_year_mean_changed_pixels_1km']
(9319, 38)
Saved: /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/Combined/combined_training_dataset.csv


In [8]:
# adding feature to handle landsat nulls
# Landsat columns to check for missingness
landsat_targets = ["nir","ndmi","blue","red","swir22","swir16","green","mndwi"]

merged_cols_lower = {c.lower(): c for c in filtered_no_pixel_flags.columns}
resolved_cols = []
for t in landsat_targets:
    if t in merged_cols_lower:
        resolved_cols.append(merged_cols_lower[t])
        continue
    alt = f"{t}__landsat"
    if alt in merged_cols_lower:
        resolved_cols.append(merged_cols_lower[alt])
        continue

missing = [t for t, c in zip(landsat_targets, resolved_cols + [None] * (len(landsat_targets) - len(resolved_cols))) if c is None]
if missing:
    raise ValueError(f"Missing expected Landsat columns in filtered_no_pixel_flags: {missing}")

# We compute landsat_present later in the engineering cell
print("Landsat columns found:", resolved_cols)
print("(landsat_present will be created in the engineering step)")

Landsat columns found: ['nir', 'NDMI', 'blue', 'red', 'swir22', 'swir16', 'green', 'MNDWI']
(landsat_present will be created in the engineering step)


In [9]:
# data engineering
# Start from normal dataset and add raster buffer columns back in
merge_keys = ["latitude", "longitude", "sample_date"]
raster_cols = [c for c in merged.columns if c.endswith("_1km")]

engineered = filtered_no_pixel_flags.merge(
    merged[merge_keys + raster_cols],
    on=merge_keys,
    how="left",
)

EPS = 1e-6

# --- Date cyclic features ---
if "sample_date" in engineered.columns:
    dates = pd.to_datetime(engineered["sample_date"], dayfirst=True, errors="coerce")
    months = dates.dt.month.fillna(1).astype(int)
    engineered["month_sin"] = np.sin(2 * np.pi * months / 12)
    engineered["month_cos"] = np.cos(2 * np.pi * months / 12)

# --- Landsat features ---
if "landsat_present" not in engineered.columns:
    landsat_cols = [c for c in ["nir","ndmi","blue","red","swir22","swir16","green","mndwi"] if c in engineered.columns]
    if landsat_cols:
        engineered["landsat_present"] = (~engineered[landsat_cols].isna().any(axis=1)).astype(int)

if all(c in engineered.columns for c in ["nir", "red"]):
    engineered["ndvi"] = (engineered["nir"] - engineered["red"]) / (engineered["nir"] + engineered["red"] + EPS)

if all(c in engineered.columns for c in ["nir", "swir22"]):
    engineered["nbr"] = (engineered["nir"] - engineered["swir22"]) / (engineered["nir"] + engineered["swir22"] + EPS)

if all(c in engineered.columns for c in ["swir16", "swir22"]):
    engineered["mineral_index"] = (engineered["swir16"] - engineered["swir22"]) / (engineered["swir16"] + engineered["swir22"] + EPS)
    engineered["swir_ratio"] = engineered["swir16"] / (engineered["swir22"] + EPS)

if all(c in engineered.columns for c in ["swir22", "nir"]):
    engineered["salinity_proxy"] = engineered["swir22"] / (engineered["nir"] + EPS)

# --- TerraClimate features ---
if all(c in engineered.columns for c in ["ppt", "pet"]):
    engineered["wb"] = engineered["ppt"] - engineered["pet"]
    engineered["eci"] = engineered["pet"] / (engineered["ppt"] + EPS)

if all(c in engineered.columns for c in ["q", "ppt"]):
    engineered["rr"] = engineered["q"] / (engineered["ppt"] + EPS)

if all(c in engineered.columns for c in ["pet", "aet"]):
    engineered["etgap"] = engineered["pet"] - engineered["aet"]

if all(c in engineered.columns for c in ["def", "ppt"]):
    engineered["dsi"] = engineered["def"] / (engineered["ppt"] + EPS)

if all(c in engineered.columns for c in ["srad", "tmax", "tmin"]):
    engineered["hri"] = engineered["srad"] * ((engineered["tmax"] + engineered["tmin"]) / 2)

# --- JRC buffer features (if present) ---
if all(c in engineered.columns for c in ["gsw_occurrence_mean_1km", "gsw_seasonality_mean_1km"]):
    engineered["water_perm"] = engineered["gsw_occurrence_mean_1km"] * (engineered["gsw_seasonality_mean_1km"] / 12)
    engineered["water_instab"] = (1 - engineered["gsw_occurrence_mean_1km"] / 100) * (engineered["gsw_seasonality_mean_1km"] / 12)

if all(c in engineered.columns for c in ["gsw_recurrence_mean_1km", "gsw_extent_mean_1km"]):
    engineered["recurrence_ratio"] = engineered["gsw_recurrence_mean_1km"] / (engineered["gsw_extent_mean_1km"] + EPS)

if "gsw_seasonality_mean_1km" in engineered.columns:
    engineered["seasonal_water"] = engineered["gsw_seasonality_mean_1km"].between(1, 9).astype(int)

# --- ESA point-based features (if present) ---
if "esa_change_count" in engineered.columns:
    engineered["esa_change_intensity"] = np.log1p(engineered["esa_change_count"])

# --- GAIA buffer interactions (if present) ---
if all(c in engineered.columns for c in ["wb", "gaia_impervious_frac_by_sample_year_1km"]):
    engineered["wb_x_impervious"] = engineered["wb"] * engineered["gaia_impervious_frac_by_sample_year_1km"]

if all(c in engineered.columns for c in ["eci", "gaia_impervious_frac_by_sample_year_1km"]):
    engineered["eci_x_impervious"] = engineered["eci"] * engineered["gaia_impervious_frac_by_sample_year_1km"]

if all(c in engineered.columns for c in ["gsw_occurrence_mean_1km", "gaia_impervious_frac_by_sample_year_1km"]):
    engineered["gsw_occ_x_impervious"] = engineered["gsw_occurrence_mean_1km"] * engineered["gaia_impervious_frac_by_sample_year_1km"]

# Save
engineered_out = os.path.join(OUTPUT_DIR, "combined_training_engineered.csv")
engineered.to_csv(engineered_out, index=False)

print("Saved:", os.path.abspath(engineered_out))
print("Engineered shape:", engineered.shape)

Saved: /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/Combined/combined_training_engineered.csv
Engineered shape: (9319, 82)


In [10]:
# Correlation matrix + highly correlated feature pairs

numeric_cols = engineered.select_dtypes(include=[np.number]).columns
corr = engineered[numeric_cols].corr()

# Full correlation matrix (can be large)
print("Correlation matrix shape:", corr.shape)
display(corr)

# List top absolute correlations (excluding self-correlation)
abs_corr = corr.abs()
np.fill_diagonal(abs_corr.values, 0)

pairs = (
    abs_corr.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b", 0: "abs_corr"})
)

# Drop duplicate pairs (A,B) vs (B,A)
pairs = pairs[pairs["feature_a"] < pairs["feature_b"]]

# Show pairs above threshold
threshold = 0.9
high_corr = pairs[pairs["abs_corr"] >= threshold].sort_values("abs_corr", ascending=False)
print(f"Highly correlated pairs (|r| >= {threshold}): {len(high_corr)}")
display(high_corr)


Correlation matrix shape: (81, 81)


,latitude,longitude,gaia_changed_ever_frac,gaia_impervious_frac_by_sample_year,gaia_recent_change_5y_frac,gaia_years_since_change_mean,gaia_transition_year_mean_changed_pixels,gsw_change,gsw_extent,gsw_occurrence,...,dsi,hri,water_perm,water_instab,recurrence_ratio,seasonal_water,esa_change_intensity,wb_x_impervious,eci_x_impervious,gsw_occ_x_impervious
latitude,1.000000,0.624468,-0.008186,-0.008266,0.034533,-0.008151,-0.008176,0.054291,-0.011862,0.029326,...,0.156446,0.137433,0.044747,0.088820,-0.123671,-0.031691,-0.074657,-0.022001,0.046817,0.058524
longitude,0.624468,1.000000,-0.041832,-0.038242,-0.010196,-0.043382,-0.041804,-0.002020,0.017278,-0.055236,...,0.039922,-0.027000,-0.183209,-0.110129,-0.110029,-0.212531,0.023355,0.033233,0.019617,0.063532
gaia_changed_ever_frac,-0.008186,-0.041832,1.000000,0.998205,0.661813,0.990930,0.999999,0.114104,-0.109352,-0.076120,...,0.004585,0.006698,-0.055838,-0.025777,-0.156020,-0.049365,-0.009109,-0.698230,0.204525,0.475113
gaia_impervious_frac_by_sample_year,-0.008266,-0.038242,0.998205,1.000000,0.662181,0.993541,0.998177,0.112587,-0.107954,-0.075351,...,0.005700,0.006101,-0.052796,-0.020204,-0.145212,-0.048761,-0.007191,-0.703732,0.209694,0.489289
gaia_recent_change_5y_frac,0.034533,-0.010196,0.661813,0.662181,1.000000,0.636831,0.662120,0.071262,-0.071484,-0.056023,...,0.015248,0.019632,-0.054133,-0.061526,-0.270178,-0.028727,-0.038268,-0.408895,0.133811,0.130653
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
seasonal_water,-0.031691,-0.212531,-0.049365,-0.048761,-0.028727,-0.048493,-0.049370,-0.267551,0.275127,0.481222,...,0.017731,0.073244,0.773830,0.567187,0.085213,1.000000,-0.034087,0.040822,-0.012350,-0.032351
esa_change_intensity,-0.074657,0.023355,-0.009109,-0.007191,-0.038268,-0.000922,-0.009191,0.041532,-0.041708,-0.067812,...,-0.033376,0.011857,-0.072055,-0.093445,-0.036940,-0.034087,1.000000,0.004128,-0.005362,-0.009898
wb_x_impervious,-0.022001,0.033233,-0.698230,-0.703732,-0.408895,-0.698259,-0.698253,-0.049239,0.031215,-0.002993,...,-0.010672,-0.100129,0.024077,-0.040151,-0.023992,0.040822,0.004128,1.000000,-0.232445,-0.588679
eci_x_impervious,0.046817,0.019617,0.204525,0.209694,0.133811,0.209616,0.204477,0.001643,0.011123,0.029546,...,0.299245,-0.077081,-0.002497,0.031533,0.006041,-0.012350,-0.005362,-0.232445,1.000000,0.241725


Highly correlated pairs (|r| >= 0.9): 54


,feature_a,feature_b,abs_corr
1877,def,etgap,1.000000
2833,esa_change_count,esa_change_intensity,1.000000
4158,gaia_changed_ever_frac_1km,gaia_transition_year_mean_changed_pixels_1km,1.000000
164,gaia_changed_ever_frac,gaia_transition_year_mean_changed_pixels,0.999999
5493,dsi,eci,0.999941
3779,gsw_seasonality_mean_1km,water_instab,0.999615
4155,gaia_changed_ever_frac_1km,gaia_impervious_frac_by_sample_year_1km,0.999557
4237,gaia_impervious_frac_by_sample_year_1km,gaia_transition_year_mean_changed_pixels_1km,0.999551
161,gaia_changed_ever_frac,gaia_impervious_frac_by_sample_year,0.998205
243,gaia_impervious_frac_by_sample_year,gaia_transition_year_mean_changed_pixels,0.998177
